In [1]:
pip install sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

# 1. Verify GPU availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device.upper()}")

# 2. Load your fine-tuned model
model_path = "/kaggle/input/notebooks/ameerhvmza/fullfinetuning/model_run_6"
model = SentenceTransformer(model_path, device=device)

# 3. Load your dataset
csv_path = "/kaggle/input/datasets/ameerhvmza/subsetprototype/final_products.csv" 
df = pd.read_csv(csv_path, low_memory=False)

# 4. Clean missing values for core identity fields
df['name'] = df['name'].fillna('')
df['brand'] = df['brand'].fillna('')
# Clean category separators (replace '>' with spaces for cleaner tokenization)
df['category_hierarchy'] = df['category_hierarchy'].fillna('').apply(lambda x: str(x).replace('>', ' '))

# 5. Format concise text (Name + Brand + Category)
# NO descriptions added here to prevent vector dilution!
combined_texts = (
    df['name'].astype(str) + 
    " | Brand: " + df['brand'].astype(str) + 
    " | Category: " + df['category_hierarchy'].astype(str)
).tolist()

# Print a sample to verify formatting
print("\n--- Sample Formatted Text (Short & Concentrated) ---")
print(combined_texts[0])
print("----------------------------------------------------\n")

# 6. Generate embeddings
print("Generating embeddings...")
embeddings = model.encode(
    combined_texts, 
    batch_size=128,          
    show_progress_bar=True, 
    convert_to_numpy=True,
    normalize_embeddings=True  # Normalizes vectors so dot-product equals cosine similarity
)

# 7. Save the updated embeddings
np.save("dataset_embeddings.npy", embeddings)

print(f"Embeddings successfully generated on {device.upper()}! Shape: {embeddings.shape}")

Using device: CUDA


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]


--- Sample Formatted Text (Short & Concentrated) ---
Freshmate Sour Cream 'n Onion Powder 80gm | Brand: Freshmate | Category: Groceries & Pets   Food Staples   Spices & Recipes
----------------------------------------------------

Generating embeddings...


Batches:   0%|          | 0/17 [00:00<?, ?it/s]

Embeddings successfully generated on CUDA! Shape: (2130, 1024)
